In [ ]:
# Dataset description ##
# M plants  = 250 total systems. 200 train, 50 test
# M x 4 component systems installed in M plants
# Each COMPONENT has 10 sensors 
# All sensors read at 1 atu for all components  stored as  [10 x 1000]
# --------------------------------------------------------------------------------------------------
# Plant 1..M
# sensor   s1      s2    s3       s4     s5     s6     s7      s8      s9     s10      Label
# COMP1  [1000] [1000] [1000]  [1000]  [1000] [1000] [1000]  [1000]  [1000]  [1000]     0/1
# COMP2  [1000] [1000] [1000]  [1000]  [1000] [1000] [1000]  [1000]  [1000]  [1000]     0/1
# COMP3  [1000] [1000] [1000]  [1000]  [1000] [1000] [1000]  [1000]  [1000]  [1000]     0/1 
# COMP4  [1000] [1000] [1000]  [1000]  [1000] [1000] [1000]  [1000]  [1000]  [1000]     0/1
# --------------------------------------------------------------------------------------------------
# Challenge: component health-state detection using multivariate sensor time-series.
# each element in the train data array is a MATLAB struct [('trajectory', 'O'), ('Label', 'O')]
# each timestamp tau has a label


#### Verify environment

In [1]:
try:
    import google.colab
    print("Running on Colab")
    from google.colab import drive
    drive.mount('/content/drive')

except:
    print("Running locally")
!ls /content/drive/MyDrive/hh_project

Mounted at /content/drive
data  prepare_data.ipynb


#### Load data

In [2]:
import scipy.io.matlab as sciom
import os
print(os.getcwd())
data = sciom.loadmat("/content/drive/MyDrive/hh_project/data/train.mat")
#data = sciom.loadmat("/home/vinayp/dev/hh_project/data/train.mat")
print(data.keys()) 


/content
dict_keys(['__header__', '__version__', '__globals__', 'Train_data'])


#### Check data structure

In [ ]:
# Whole train data
train_d = data['Train_data']
print(f"ad shape:{train_d.shape},ad dtype:{train_d.dtype}") 

# Get sensor values of all plants
iplant_d = train_d['trajectory']
print(f"apd shape:{iplant_d.shape}, apd dtype:{iplant_d.dtype}") 

#Get labels for all plants
label_d = train_d['Label']
print(f"ald shape:{label_d.shape}, ald dtype:{label_d.dtype}") 

# Get single plant data
plant1 = iplant_d[0][0]
print(f"spd shape:{plant1.shape}, spd dtype:{plant1.dtype}") 

# Get single plant labels
label1 = label_d[0][0]  #(1,4)
print(f"sld shape:{plant1.shape}, sld dtype:{plant1.dtype}") 

# Get single component data
comp1 = plant1[0][0]
print(f"scd shape:{comp1.shape}")
# Get single component label
clabel1 = label1[0:]
print(f"cld shape:{clabel1.shape}")
print(f"cld shape:{clabel1.shape[1]}")

import numpy as np

# Check abnormality time
for t in range(label1.shape[0]):
    y = label1[t]
    idx = np.where(y == 1)[0]
    if len(idx) > 0:
        print(f"component {t} abnormal at t = {idx[0]}")
    else:
        print(f"component {t} was never abnormal")



ad shape:(1, 200),ad dtype:[('trajectory', 'O'), ('Label', 'O')]
apd shape:(1, 200), apd dtype:object
ald shape:(1, 200), ald dtype:object
spd shape:(1, 4), spd dtype:object
sld shape:(1, 4), sld dtype:object
scd shape:(10, 1000)
cld shape:(4, 1000)
cld shape:1000
component 0 abnormal at t = 944
component 1 abnormal at t = 932
component 2 was never abnormal
component 3 abnormal at t = 981


#### Access one plant data. 4 components x 10 sensors x 1000 readings

In [ ]:
import plotly.graph_objects as pltg
from plotly.subplots import make_subplots
import plotly.io as pio


pio.renderers.default = "notebook"
fig = make_subplots(rows=10, cols=4, shared_xaxes=True,vertical_spacing=0.08, subplot_titles=("sensor reading", "label"))

plant_i = train_d['trajectory'][0,0] # Access whole plat 1

id = 1
# component : A nested python list containing numpy arrays
print(plant_i.shape) # Accessing components (1,4)
for i, component in enumerate(plant_i):
    print(component[0][0])
    fig.add_trace(
        pltg.Scatter(
            y = component[0][id],
            mode = "lines",
            name = "sensor",
            hovertemplate='Time: %{x}<br>Value:{y}<extra></extra>'
        ),
        row = id,
        col = 1
    )
    fig.add_trace(
        pltg.Scatter(
            y = label[id],
            mode = "lines",
            name = "label",
            hovertemplate='Time: %{x}<br>Value:{y}<extra></extra>'
        ),
        row = id,
        col = 1
    )
id=id+1
    # Axis labels
fig.update_xaxes(title_text="Time (atu)", row=2, col=1)
fig.update_yaxes(title_text="sensor_reading", row=1, col=1)
fig.update_yaxes(title_text="label", row=2, col=1)

fig.show()

#### Access ONE sensor in ONE component in ONE plant

In [ ]:
plants = train_d[0,0]  # [('trajectory', 'O'), ('Label', 'O')]
components = plants['trajectory']  # Object, shape (1,4)
componentx = components[0,0]  # float64, shape (10,1000)
sensorx = componentx[2,:] # float64, shape (10,1000)

print(f"sensor:\nshape:{sensorx.shape}, \n min: {sensorx.min()}, \n max:{sensorx.max()}") 

labelx = plants['Label']
labelc1 = labelx[2,:]
print(f"label:{labelc1.shape}")

#### Better view

In [ ]:
import plotly.graph_objects as pgo
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "notebook"

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, subplot_titles=("Sensor Reading", "label"))

fig.add_trace(
    pgo.Scatter(
        y = sensorx,
        mode = "lines",
        name = "sensor",
        hovertemplate='Time: %{x}<br>Value:{y}<extra></extra>'
    ),
    row=1,
    col=1
)

fig.add_trace(
    pgo.Scatter(
        y = labelc1,
        mode = 'lines',
        name = 'Label',
        hovertemplate = 'Time: %{x}<br>Label:%{y}<extra></extra>'
    ),
    row = 2,
    col = 1
)

fig.update_layout(
    height = 700,
    title = "Sensor and Label visualization",
    hovermode="x unified"
)

# Axis labels
fig.update_xaxes(title_text="Time (atu)", row=2, col=1)
fig.update_yaxes(title_text="sensor_reading", row=1, col=1)
fig.update_yaxes(title_text="label", row=2, col=1)

fig.show()

##### Experiment: Train a model to predict health of component or plant and identify malfunction